# Step 2: Task Deployment
2nd script of "Integrating LLMs with Process Mining to Enhance Internal Audit Insights"

Purpose:<br>
Runs pm4py to produce abstractions and executes RISEN‑structured prompts via Azure OpenAI.

## Import libraries

PM4PY requires graphviz: https://kb.databricks.com/libraries/install-pygraphviz<br>
https://stackoverflow.com/questions/77353022/need-help-to-install-graphviz-in-databricks

In [0]:
%sh 
sudo add-apt-repository universe
sudo apt-get update

In [0]:
%sh 
sudo apt-get install -y python3-dev graphviz libgraphviz-dev pkg-config

In [0]:
# import and options
import re
import numpy as np
from delta.tables import *
import datetime
import time
import pandas as pd

# pyspark
from pyspark.sql.functions import (col, when, trim, concat, concat_ws, abs, substring, lit, year, lower, isnull, length, date_add, add_months, date_trunc, month, last_day, countDistinct, sum as spark_sum, min as spark_min, max as spark_max, first, last, count, round as spark_round, datediff, create_map, to_timestamp, expr, udf, lag, dayofmonth, year, month, split, lpad, left, right, timestamp_diff, mean as spark_mean)
from pyspark.sql.types import FloatType, TimestampType, IntegerType, StringType

# GraphFrames
from graphframes import GraphFrame
# PM4Py - to be installed on cluster 
import pm4py
import graphviz

# for LLM
import tiktoken
import json

# Azure OpenAI + AAD auth
from openai import AzureOpenAI, OpenAI
from azure.identity import ClientSecretCredential

## Requirements/Parameters
Path for import and export table in Unity Catalog

In [0]:
# TODO AUDIT NO for labelling of results
audit_no = "cz_master"

# TODO path to Unity Catalog Schema: for import and export of structured data
# Format "<unity_catalog>.<schema>"
unity_catalog_schema = "<unity_catalog>.<schema>"

# TODO table name of Step 1 results
table_pm_events = "<table>"

# filepath for file export
f_path_base = f"/Volumes/t_ca_audit/landing_zone/landing_ext_volume/export/{audit_no}"

# ========== LLM & Vector Search configuration ==========
# LLM client
oai_endpoint = "https://<oai_endpoint>.openai.azure.com/openai/v1/" # select v1 API
oai_model = "gpt-4o" # deployment name for GPT-4o

# ========== Authentication via Databricks secrets ==========
# TODO replace with your own secret scope
# TENANT_ID is generally not sensitive; client ID & secret come from secret scope.
tenant_id  = "xxx"  # same tenant as original notebook

# IMPORTANT:
#   - Replace "credentials" with your actual secret scope name if different.
#   - Ensure clientId and SP-Password are defined in that scope.
client_id  = dbutils.secrets.get(scope="credentials", key="clientId")
client_secret = dbutils.secrets.get(scope="credentials", key="SP-Password")

# Create AAD credential (Entra ID Authentication)
cred = ClientSecretCredential(
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

token_provider = lambda: cred.get_token("https://cognitiveservices.azure.com/.default").token

# ========== TIKTOKEN ========== 
# calculate the number of tokens in the string
# context window of GPT-4o is 128k
encoding = tiktoken.encoding_for_model(oai_model)

## Process Mining-Functions
Documentation:
- https://processintelligence.solutions/pm4py
- Tutorials: https://processintelligence.solutions/pm4py/examples
- API-documentation: https://processintelligence.solutions/static/api/2.7.17/index.html
- https://pm4py-source.readthedocs.io/en/stable/py-modindex.html

format_dataframe: https://processintelligence.solutions/static/api/2.7.17/generated/pm4py.utils.format_dataframe.html#pm4py-utils-format-dataframe 

In [0]:
def pm4py_convert_spark(df_spark, case_id, activity_key, timestamp_key, resource_key, dict_rename):
    """
    Converts a PySpark dataframe to a Pandas dataframe in PM4Py-format.

    Parameters:
    ------------
    df_spark: PySpark dataframe to be converted
    case_id: string for column representing the case id -> case:concept:name
    activity_key: string for column representing the activity -> concept:name
    timestamp_key: string for column representing the timestamp -> time:timestamp
    resource_key: string for column representing the resource (currently not used) -> org:resource
    dict_rename: dictionary with column names to be renamed for integration as attribute 
    (e.g. specify that a column describes a case-level attribute >> {'clientID': 'case:clientID'})

    Return:
    -----------
    Pandas dataframe with all original columns and add. PM4Py-columns renamed as mentioned above

    """
    if len(dict_rename) > 0:
        df = df_spark.withColumnsRenamed(dict_rename)
    else:
        df = df_spark
    
    return (
        pm4py.format_dataframe(
        # PySpark to Pandas
        df.toPandas(), 
        case_id=case_id, 
        activity_key=activity_key, 
        timestamp_key=timestamp_key)
        )

In [0]:
def pm4py_dfg(pm4py_df, audit_no, graph_descr, case_id_key='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp'):
    """
    Creates a Directly-follows graph (DFG) based on a PM4Py dataframe.

    Parameters:
    --------------
    pm4py_df: Pandas dataframe in PM4Py format with standard column names
    case_id_key: string for column representing the case id -> case:concept:name
    activity_key: string for column representing the activity -> concept:name
    timestamp_key: string for column representing the timestamp -> time:timestamp
    audit_no: string for file name of generated graph
    graph_descr: string for file name of generated graph

    Returns:
    -----------
    Tuple of start actvities and end activities as dicts.
    """
    dfg, start_activities, end_activities = pm4py.discover_dfg(
        pm4py_df, 
        case_id_key='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp')
    
    #pm4py.view_dfg(dfg, start_activities, end_activities, format='svg')

    pm4py.save_vis_dfg(
        dfg=dfg, start_activities=start_activities, end_activities=end_activities, 
        file_path=f"{f_path_base}_{graph_descr}.png",
        graph_title=graph_descr)
    
    return start_activities, end_activities

In [0]:
def pm4py_petri_net(pm4py_df, audit_no, graph_descr, 
                    case_id_key='case:concept:name', activity_key='concept:name', timestamp_key='time:timestamp', 
                    noise_threshold=0.0,
                    dependency_threshold=0.5, and_threshold= 0.65, loop_two_threshold=0.5):
    """
    Discover Petri net using Inductive Miner algorithm.
    Best for handling noise and guaranteeing sound process models.


    Parameters:
    --------------
    pm4py_df: Pandas dataframe in PM4Py format with standard column names
    audit_no: string for file name of generated graph
    graph_descr: string for file name of generated graph
    case_id_key: string for column representing the case id -> case:concept:name
    activity_key: string for column representing the activity -> concept:name
    timestamp_key: string for column representing the timestamp -> time:timestamp

    For INDUCTIVE MINER:
    noise_threshold: controls how aggressively the Inductive Miner—infrequent variant filters out low-frequency (i.e., “noisy”) behavior before detecting cuts and building the model (default: 0.0)
        >> 0.0 (default) → no noise filtering: the miner keeps all directly-follows relations from the log
        >> Higher values (e.g., 0.05, 0.1, 0.2) → stronger filtering: increasingly infrequent edges/relations are removed from the directly-follows graph before mining, leading to a simpler, cleaner model that generalizes better but may “forget” rare behavior (lower fitness on those traces).

    For HEURISTIC MINER:
    dependency_threshold: minimum dependency measure required to keep a relation A→B in the Heuristics Net (filters weak/ambiguous directly follows edges).   
        >> default: 0.5
        >> Increase (e.g., 0.6–0.8) to remove noisy/weak edges; decrease (e.g., 0.3–0.4) to keep more relations and increase fitness (at the cost of simplicity/precision)
    and_threshold: Threshold to decide parallelism (AND)—controls how confidently the miner declares two successors as parallel rather than exclusive.
        >> default: 0.65
        >> Increase to be conservative (fewer parallel gateways), decrease to accept more parallel constructs.
    loop_two_threshold: Sensitivity for detecting length-two loops (patterns like A↔B, e.g., ABAB…).
        >> default: 0.5
        >> Raise when you only want clear 2-loops; lower to accept weaker evidence for 2-loops.
    

    Returns:
    -----------
    Dictionary with tuples (Petri net, initial marking, final marking) of the generated Petri nets
    """
    #f_path_base = f"/Volumes/t_ca_audit/landing_zone/landing_ext_volume/export/{audit_no}_{graph_descr}"
    f_path_interim = f"{f_path_base}_{graph_descr}"

    # INDUCTIVE Miner
    # returns Petri net, marking, marking
    i_net, i_im, i_fm = pm4py.discover_petri_net_inductive(
        pm4py_df,
        activity_key=activity_key,
        case_id_key=case_id_key,
        timestamp_key=timestamp_key,
        noise_threshold=noise_threshold,
        )
    
    f_path_ind = f"{f_path_interim}_inductive.png"

    pm4py.save_vis_petri_net(
        petri_net=i_net, initial_marking=i_im, final_marking=i_fm, 
        file_path=f_path_ind,
        graph_title=f"{graph_descr} with Inductive miner and noise threshold {noise_threshold}"
        )
    print(f"Petri-Net with Inductive Miner saved under {f_path_ind}")
    
    # HEURISTIC Miner
    # returns Petri net, marking, marking
    h_net, h_im, h_fm = pm4py.discover_petri_net_heuristics(
        pm4py_df,
        activity_key=activity_key,
        case_id_key=case_id_key,
        timestamp_key=timestamp_key,
        dependency_threshold=dependency_threshold, 
        and_threshold=and_threshold, 
        loop_two_threshold=loop_two_threshold, 
        )
    
    f_path_heu = f"{f_path_interim}_heuristic.png"

    pm4py.save_vis_petri_net(
        petri_net=h_net, initial_marking=h_im, final_marking=h_fm, 
        file_path=f_path_heu,
        graph_title=f"{graph_descr} with Heuristic miner and noise threshold{noise_threshold}"
        )
    print(f"Petri-Net with Heuristic Miner saved under {f_path_heu}")

    dict_return = {"inductive": (i_net, i_im, i_fm), "heuristic": (h_net, h_im, h_fm)}

    return dict_return

# Create event log

## Load data

In [0]:
df_in = spark.table(unity_catalog_schema + "." + table_pm_events)

total_events = df_in.count()
total_ibl = df_in.select("ibl_trace_id").distinct().count()
total_obl = df_in.select("obl_trace_id").distinct().count()
total_ibl_obl = df_in.select("ibl_obl_id").distinct().count()

print(f"Total rows: {total_events:,} >> Distinct IBLs: {total_ibl:,} >> Distinct OBLs: {total_obl:,} >> distinct IBL/OBL {total_ibl_obl:,}")
#df_in.printSchema()

## System data to Event Log
Columns in use:<br>

PROCESS:<br>
 |-- scwm_ordim_c_mandt: string (nullable = true)<br>
 |-- scwm_ordim_c_lgnum: string (nullable = true)<br>

 CASE:<br>
 |-- ibl_obl_id: string (nullable = true)<br>
 |-- sapapo_matkey_matnr: string (nullable = true)<br>
 |-- sapapo_mattxt_maktx: string (nullable = true)<br>
 |-- scwm_ordim_c_wdatu: timestamp (nullable = true)<br>
 |-- qty_unit: string (nullable = true)<br>

 EVENT/ACTIVITY:<br>
 |-- timestamp: timestamp (nullable = true)<br>
 |-- process_id-category-type: string (nullable = true)<br>
 |-- storage_type_id-text: string (nullable = true)<br>
 |-- qty_nistm_actual: decimal(38,18) (nullable = true)<br>


In [0]:
# dictionary to rename to be considered as case-attributes
# remaining cols are automatically event-attributes
dict_case = {
    "sapapo_matkey_matnr": "case:material_no", 
    "sapapo_mattxt_maktx": "case:material_descr",
    "scwm_ordim_c_wdatu": "case:gr_date", 
    "qty_unit": "case:unit"}

**Event log and further analyses to be separate by scope: either Process- or Location-scope!**

In [0]:
# Event log for Process-perspective
df_log_process = pm4py_convert_spark(
  df_in
  .select(
      # CASE
      "ibl_obl_id", "sapapo_matkey_matnr", "sapapo_mattxt_maktx", "scwm_ordim_c_wdatu", "qty_unit", 
      # EVENT
      "timestamp", "process_id-category-type", "storage_type_id-text", "qty_nistm_actual"
      ),
  case_id="ibl_obl_id",
  activity_key="process_id-category-type",
  timestamp_key="timestamp",
  resource_key="",
  dict_rename=dict_case
  )

In [0]:
# Event log for Location-perspective
df_log_location = pm4py_convert_spark(
  df_in
  .select(
      # CASE
      "ibl_obl_id", "sapapo_matkey_matnr", "sapapo_mattxt_maktx", "scwm_ordim_c_wdatu", "qty_unit",
      # EVENT
      "timestamp", "process_id-category-type", "storage_type_id-text", "qty_nistm_actual"
      ),
  case_id="ibl_obl_id",
  activity_key="storage_type_id-text",
  timestamp_key="timestamp",
  resource_key="",
  dict_rename=dict_case
  )

# Process Discovery 

## Directly-follows graph (DFG)

In [0]:
# Process-perspective
pm4py_dfg(pm4py_df=df_log_process, audit_no=audit_no, graph_descr="01a_dfg_process")

# Location-perspective
pm4py_dfg(pm4py_df=df_log_location, audit_no=audit_no, graph_descr="01b_dfg_location")

## Filter top k variants

In [0]:
# calculate total number of variants
# returns dict with variant as tuple and count of traces
process_variants_count = pm4py.get_variants(df_log_process) 
location_variants_count = pm4py.get_variants(df_log_location) 

print(f"Total number of variants in PROCESS-perspective: {len(process_variants_count):,} and in LOCATION-perspective: {len(location_variants_count):,}")

In [0]:
# Top k variants >> create DFGs and statistics
list_k = [
    10, 20, 50, 100, 250, 
    #500, 750, 1000, 2500, 5000, 7500, 10000, 12500, 15000, 17500, 20000, 21000
    ]

# collect shares of covered events
list_process_events_share = []
list_location_events_share = []

# collect shares of covered cases
list_process_case_share = []
list_location_case_share = []

#dictionaries to collect filtered event logs
dict_top_k_log_process = {}
dict_top_k_log_location = {}

for k in list_k:
    # Process-perspective
    if k <= len(process_variants_count):
        # filter log
        filtered_log_process = pm4py.filter_variants_top_k(df_log_process, k)
        dict_top_k_log_process[k] = filtered_log_process 
        # calculate statistics
        top_k_process_cases = filtered_log_process["ibl_obl_id"].nunique()
        top_k_process_events = filtered_log_process.shape[0]
        list_process_events_share.append(np.round(top_k_process_events / total_events, 3))
        list_process_case_share.append(np.round(top_k_process_cases / total_ibl_obl, 3))
        print(f"DFG in PROCESS-perspective top {k:,} variants contains:")
        print(f">> Total Cases: {top_k_process_cases:,} >> {top_k_process_cases / total_ibl_obl:.1%} of total IBL_OBL_IDs")
        print(f">> Total Events: {top_k_process_events:,} >> {top_k_process_events / total_events:.1%} of total Events")
        start_act_process, end_act_process = pm4py_dfg(pm4py_df=filtered_log_process, audit_no=audit_no, graph_descr=f"02a_dfg_top_{str(k).zfill(5)}_process")
    else:
        list_process_events_share.append(1)
        list_process_case_share.append(1)

    # Location-perspective
    if k <= len(location_variants_count):
        # filter log
        filtered_log_location = pm4py.filter_variants_top_k(df_log_location, k)
        dict_top_k_log_location[k] = filtered_log_location
        # calculate statistics
        top_k_location_cases = filtered_log_location["ibl_obl_id"].nunique()
        top_k_location_events = filtered_log_location.shape[0]
        list_location_events_share.append(np.round(top_k_location_events / total_events, 3))
        list_location_case_share.append(np.round(top_k_location_cases / total_ibl_obl, 3))
        print(f"DFG in LOCATION-perspective top {k:,} variants contains:")
        print(f">> Total Cases: {top_k_location_cases:,} >> {top_k_location_cases / total_ibl_obl:.1%} of total IBL_OBL_IDs")
        print(f">> Total Events: {top_k_location_events:,} >> {top_k_location_events / total_events:.1%} of total Events")
        start_act_location, end_act_location = pm4py_dfg(pm4py_df=filtered_log_location, audit_no=audit_no, graph_descr=f"02b_dfg_top_{str(k).zfill(5)}_location")
    else:
        list_location_events_share.append(1)
        list_location_case_share.append(1)


# consolidate statistics
df_top_k = pd.DataFrame(
    {
        "top_k": list_k, 
        "process_events_share": list_process_events_share, "location_events_share": list_location_events_share,
        "process_case_share": list_process_case_share, "location_case_share": list_location_case_share
        }, 
    columns = ["top_k", "process_events_share", "location_events_share", "process_case_share", "location_case_share"]
    )

## Petri-Net Process Model

In [0]:
dict_pnet = pm4py_petri_net(
    pm4py_df=df_log_location, audit_no=audit_no, graph_descr="03b_petri_location", 
    # for Inductive Miner: 
    noise_threshold=0.2, # increase from 0
    # for Heuristic miner
    dependency_threshold=0.9, # increase from default = 0.5 to remove noisy edges
    and_threshold= 0.95, # increase from default = 0.65 to have fewer parallel gateways
    )

Conclusion for Petri Nets:
> Inductive Miner offers good overview but is incorrect (see variants with INDC)<br>
> Heuristic Miner way too complicated. No advantage over DFG

## Create Abstractions

Ref: https://processintelligence.solutions/static/api/2.7.17/api.html#llm-integration-pm4py-llm

### DFG

In [0]:
# LOCATION: Abstraction for DFG
abstr_dfg_loc = pm4py.llm.abstract_dfg(
  log_obj=df_log_location, max_len=100000000, include_performance=False, relative_frequency=False, response_header=True
  )

abstr_dfg_loc_tokens = encoding.encode(abstr_dfg_loc)
print(f"Abstraction of DFG consists of {len(abstr_dfg_loc):,} characters and {len(abstr_dfg_loc_tokens):,} tokens.")

In [0]:
# PROCESS: Abstraction for DFG
abstr_dfg_proc = pm4py.llm.abstract_dfg(
  log_obj=df_log_process, max_len=100000000, include_performance=False, relative_frequency=False, response_header=True
  )

abstr_dfg_proc_tokens = encoding.encode(abstr_dfg_proc)
print(f"Abstraction of DFG consists of {len(abstr_dfg_proc):,} characters and {len(abstr_dfg_proc_tokens):,} tokens.")

### Variants (location perspective)

In [0]:
# Abstraction for Variants
abstr_variants = pm4py.llm.abstract_variants(
  log_obj=df_log_location, max_len=100000000, include_performance=False, relative_frequency=False, response_header=True
  )

abstr_variants_tokens = encoding.encode(abstr_variants)
print(f"Abstraction of Variants consists of {len(abstr_variants):,} characters and {len(abstr_variants_tokens):,} tokens.")

In [0]:
# select BOTTOM 100 process variants out of total variants
abstr_variants_split = abstr_variants.split("\n")
bottom_100_abstr_variants_list = (
    abstr_variants_split[0:2] # first 2 rows as header
    + abstr_variants_split[-100:] # last 100 rows
    )
#len(bottom_100_abstr_variants)

bottom_100_abstr_variants = "\n".join(bottom_100_abstr_variants_list)
#print(bottom_100_abstr_variants)

bottom_100_abstr_variants_tokens = encoding.encode(bottom_100_abstr_variants)
print(f"Abstraction of Variants consists of {len(bottom_100_abstr_variants):,} characters and {len(bottom_100_abstr_variants_tokens):,} tokens.")

In [0]:
# simulate abstraction for TOP K variants
for k in dict_top_k_log_location:
    # Abstraction for Variants
    top_k_abstr_variants = pm4py.llm.abstract_variants(
        log_obj=dict_top_k_log_location[k], max_len=100000000, include_performance=False, relative_frequency=False, response_header=True
        )
    
    top_k_abstr_variants_tokens = encoding.encode(top_k_abstr_variants)
    print(f"Abstraction of TOP {k} variants consists: {len(top_k_abstr_variants):,} characters and {len(top_k_abstr_variants_tokens):,} tokens.")

In [0]:
top_100_abstr_variants = pm4py.llm.abstract_variants(
        log_obj=dict_top_k_log_location[100], max_len=100000000, include_performance=False, relative_frequency=False, response_header=True
        )

# LLM-integration

OAI API-reference: https://ai.azure.com/doc/azure/ai-foundry/openai/supported-languages?pivots=programming-language-python&tid=505cca53-5750-4134-9501-8d52d5df3cd1

API-version: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/api-version-lifecycle?view=foundry-classic&tabs=python

How-to Responses API: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry-classic&tabs=python-key <br>
**Responses API**: The Responses API is a newer Azure OpenAI feature that requires specific Azure RBAC permissions. Your service principal doesn't have the necessary permission to call the /openai/v1/responses endpoint.

How-to Chat Completion: https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/chatgpt?view=foundry-classic&tabs=python-secure

Token limit for GPT-4o is 8,192 tokens:
https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/chatgpt?view=foundry-classic&tabs=python-secure#manage-conversations

## Initialize LLM-client

In [0]:
# LLM-client
try:
    llm_client = OpenAI(
        base_url =oai_endpoint,
        api_key = token_provider, 
        max_retries = 1
        )

    print("LLM-client successfully initialized.")
except Exception as e:
    print("LLM-client init failed. Error:", e)
    llm_client = None

In [0]:
def call_llm(messages):
    response_content = ""
    tokens_prompt = 0
    tokens_completion = 0
    error_code = None
    # attempt call
    try:
        # Chat completion 
        response = llm_client.chat.completions.create(
            model=oai_model,
            temperature=0,
            messages=messages,
        )
        #content = response.choices[0].message.content
        response_json = json.loads(response.model_dump_json(indent=2))

        # select information
        response_content = response_json["choices"][0]["message"]["content"]
        finish_reason = response_json["choices"][0]["finish_reason"]
        tokens_completion = response_json["usage"]["completion_tokens"]
        tokens_prompt = response_json["usage"]["prompt_tokens"]

        if finish_reason == "stop":
            print("Success: response completed.")
        elif finish_reason == "length":
            print("Warning: response truncated due to token limit.")
        elif finish_reason == "content_filter":
            print("Warning: response filtered due to content filter.")
        else:
            print("Warning: response in progress or incomplete due to unknown reason.")

        # Responses API: not working with current principal
        """response = llm_client.responses.create(
            model=oai_model,
            input= "This is a test"
            )"""
    except Exception as e:
        print("Attempt failed. Error:", e)
        """if hasattr(e, "status_code"):
            error_code = e.status_code
        elif hasattr(e, "response") and hasattr(e.response, "status_code"):
            error_code = e.response.status_code
        else:
            error_code = "unknown"
            """

    return response_content, tokens_prompt, tokens_completion

## Prompt modules

In [0]:
# Prompt modules
# ======= ROLE =======
system_1_role = (
        "### ROLE "
        "You are an expert in Process Mining and an experienced analyst of logistics and warehouse material-flow processes. "
        "You analyze processes spanning the material flow in a warehouse – from goods receipt to outbound dispatch. "
        )

# ======= INSTRUCTIONS =======
user_1_d1 = (
        "### INSTRUCTIONS "
        "Your task is to describe the underlying warehouse material-flow processes represented in the Process Mining data provided. "
        )
user_1_d2 = (
        "### INSTRUCTIONS "
        "Your task is to describe the process variants of the underlying warehouse material-flow represented in the Process Mining data provided. "
        )
user_1_c1 = (
        "### INSTRUCTIONS "
        "Your task is to analyze the Process Mining data provided by using your domain expertise to identify and describe the most significant anomalies present in the warehouse material-flow processes. "
        )
user_1_c2 = (
        "### INSTRUCTIONS "
        "Your task is to analyze the least frequent process variants provided by using your domain expertise to identify and describe the most significant anomalies present in the warehouse material-flow processes. "
        )
user_1_v1 = (
        "### INSTRUCTIONS "
        "Your task is to perform semantic clustering on the process variants provided. "
        "Summarize the clustered process variants, clearly describing the main characteristics and patterns within each cluster. "
        )
        
# Context
user_2_context = (
        "Process Mining Context & Definitions: "
        "- The event log represents process executions. Each event corresponds to the execution of a single process activity or step and is related to a single case or process instance. "
        "- A case is the chronological sequence of events describing one complete process run. "
        "- A trace/variant is the ordered list of activities that occur within a case. "
        "- The directly-follows graph (DFG) visualizes activities (nodes) and their directly-following relationships (edges), showing how the process typically flows. "
        "- Please consider special patterns: In warehouse process mining data, storage and retrieval at the same location (particularly in storage areas) are recorded as separate events, creating self-loops in activity logs. This is a result of data capture practices and does not indicate physical anomalies. "
        )

# Few-shot learning with C1 & C2
user_example_1 = (
        "### EXAMPLE 1 "
        "Sequence of events: Inbound Deconsolidation -> Small Parts -> Pallet Rack "
        )
assistant_example_1 = (
        "Anomaly: Putaway material from Small Parts storage in Pallet Racks without consolidation and repacking is not appropriate. "
        )
user_example_2 = (
        "### EXAMPLE 2 "
        "Sequence of events: Inbound Receiving -> Outbound "
        )
assistant_example_2 = (
        "Anomaly: A direct inbound-to-outbound pass is an exceptional process. "
        )
user_example_3 = (
        "### EXAMPLE 3 "
        "Sequence of events: Inbound -> Bulk storage -> Inbound "
        )
assistant_example_3 = (
        "Anomaly: In an efficiently managed warehouse, materials should not need to be returned from the storage area to the inbound area. "
        ) 
user_example_4 = (
        "### EXAMPLE 4 "
        "Sequence of events: Putaway -> Stock removal -> Putaway "
        )
assistant_example_4 = (
        "Anomaly: In an efficiently managed warehouse, materials should not be stored again after stock removal. "
        )        

# Input Data
user_3_input = (
        "### INPUT DATA "
        "The following Process Mining abstractions represent the same underlying warehouse process, with events representing material movements between locations: "
        )
# D1
user_4a_d1_corpus = abstr_dfg_loc
user_4b_d1_input = (
        "Additionally, consider the following Process Mining abstractions, which represent the same underlying warehouse process, with events representing the process types for material movements: "
        )
user_4c_d1_corpus = abstr_dfg_proc

user_4_d2_corpus = top_100_abstr_variants

# C1
user_4a_c1_corpus = user_4a_d1_corpus
user_4b_c1_input = user_4b_d1_input
user_4c_c1_corpus = user_4c_d1_corpus

user_4_c2_corpus = bottom_100_abstr_variants
user_4_v1_corpus = top_100_abstr_variants

# ======= STEPS =======
user_5_d1_steps = (
        "### STEPS "
        "- Carefully review and understand the Process Mining data, consisting of both DFGs. "
        "- Compare both abstractions to identify how they relate and how they represent 2 perspectives on the same executions. "
        "- Summarize your understanding by describing the material-flow concept captured in the data. "
        )
user_5_d2_steps = (
        "### STEPS "
        "- You are provided with the most frequent process variants. "
        "- Carefully review and understand the process variants provided. "
        "- Identify and summarize patterns in the process variants. "
        "- Describe the sequence patterns of material movements (events) between locations in the warehouse. "
        )
user_5_c1_steps = (
        "### STEPS "
        "- Carefully review and understand the Process Mining data, consisting of both DFGs. "
        "- Compare both abstractions to identify how they relate and how they represent 2 perspectives on the same executions. "
        "- Apply your domain knowledge to spot and document any possible anomalies, unusual sequences, or deviations that become evident from the data. "
        "- Consider the specific patterns described in Process Mining Context. "
        "- Summarize your findings on potential anomalies with a conclusion highlighting the material movements affected and insights about the material-flow process. "
        )
user_5_c2_steps = (
        "### STEPS "
        "- You are provided with the least frequent process variants. "
        "- Carefully review and understand the process variants provided. "
        "- Apply your domain knowledge to spot and document any possible anomalies, unusual sequences, or deviations that become evident from the data. "
        "- Consider the specific patterns described in Process Mining Context. "
        "- Summarize your findings on potential anomalies with a conclusion highlighting the material movements affected and insights about the material-flow process. "
        )
user_5_v1_steps = (
        "### STEPS "
        "- Carefully review and understand the process variants provided "
        "- Apply semantic clustering techniques to group similar process variants together. "
        "- Consider the specific patterns described in Process Mining Context. "
        "- For each resulting cluster, analyze and summarize its main characteristics and patterns. Reference specific behaviors, sequences, or features that distinguish each cluster, using only the process mining abstractions provided. "
        )

# ======= END GOAL =======
output_length_words = 750

system_2_d1_objective = (
        "### END GOAL "
        "A well-structured explanation of the warehouse material-flow concept, derived solely from the Process Mining abstractions provided. "
        f"Provide a clear, concise description of the material-flow process with a maximum length of {output_length_words} words. "
        )
system_2_d2_objective = (
        "### END GOAL "
        "A well-structured explanation of the patterns identified in the most frequent process variants, derived solely from the Process Mining abstractions provided.  "
        f"Provide a clear, concise description of the material-flow process with a maximum length of {output_length_words} words. "
        )
system_2_c1_objective = (
        "### END GOAL "
        "A well-structured summary of the possible anomalies, derived solely from the descriptions provided. "
        "Be thorough in your explanation, referencing specific patterns or irregularities detected. "
        f"Provide a clear, concise explanation of your findings with a maximum length of {output_length_words} words. "
        )
system_2_c2_objective = (
        "### END GOAL "
        "A well-structured summary of the possible anomalies, derived solely from the descriptions provided. "
        "Be thorough in your explanation, referencing specific patterns or irregularities detected. "
        f"Provide a clear, concise explanation of your findings with a maximum length of {output_length_words} words. "
        )
system_2_v1_objective = (
        "### END GOAL "
        "Generate a well-structured summary of the semantic clusters derived from the process variants provided. "
        "This summary must thoroughly explain the main characteristics and patterns within each cluster. "
        f"Provide a clear, concise explanation of your results with a maximum length of {output_length_words} words. "
        )

# ======= NARROWING =======
system_3_narrowing = (
        "### NARROWING "
        "- Stay strictly grounded in the Process Mining data and do not invent details. " 
        "- If something remains unclear from the data, explicitly state your uncertainty. "
        "- Apply your domain knowledge to understand the events and the process. "
        "- The explanation must be understandable to business experts and auditors."
        )

## Configure and Run

In [0]:
# ======= D1: Provide a description of the processes underlying this data.
prompt_d1 = [
            {"role": "system", "content": system_1_role},
            {"role": "user", "content": user_1_d1},
            {"role": "user", "content": user_2_context},
            {"role": "user", "content": user_3_input},
            {"role": "user", "content": user_4a_d1_corpus},
            {"role": "user", "content": user_4b_d1_input},
            {"role": "user", "content": user_4c_d1_corpus},
            {"role": "user", "content": user_5_d1_steps},
            {"role": "system", "content": system_2_d1_objective},
            {"role": "system", "content": system_3_narrowing},
        ]

resp, t_prompts, t_completions = call_llm(prompt_d1)
print(f"LLM response based on prompt with {t_prompts:,} tokens and response of {t_completions:,} tokens.")
print(resp)

In [0]:
# ======= D2: Provide a detailed description of the most frequent process variants.
# TOP 100
prompt_d2 = [
            {"role": "system", "content": system_1_role},
            {"role": "user", "content": user_1_d2},
            {"role": "user", "content": user_2_context},
            {"role": "user", "content": user_3_input},
            {"role": "user", "content": user_4_d2_corpus},
            {"role": "user", "content": user_5_d2_steps},
            {"role": "system", "content": system_2_d2_objective},
            {"role": "system", "content": system_3_narrowing},
        ]

resp, t_prompts, t_completions = call_llm(prompt_d2)
print(f"LLM response based on prompt with {t_prompts:,} tokens and response of {t_completions:,} tokens.")
print(resp)

In [0]:
# ======= C1: Identify the main anomalies in this data according to your domain knowledge of the process.
prompt_c1 = [
            {"role": "system", "content": system_1_role},
            {"role": "user", "content": user_1_c1},
            {"role": "user", "content": user_2_context},
            {"role": "user", "content": user_example_1},
            {"role": "assistant", "content": assistant_example_1},
            {"role": "user", "content": user_example_2},
            {"role": "assistant", "content": assistant_example_2},
            {"role": "user", "content": user_example_3},
            {"role": "assistant", "content": assistant_example_3},
            {"role": "user", "content": user_example_4},
            {"role": "assistant", "content": assistant_example_4},
            {"role": "user", "content": user_3_input},
            {"role": "user", "content": user_4a_c1_corpus},
            {"role": "user", "content": user_4b_c1_input},
            {"role": "user", "content": user_4c_c1_corpus},
            {"role": "user", "content": user_5_c1_steps},
            {"role": "system", "content": system_2_c1_objective},
            {"role": "system", "content": system_3_narrowing},
        ]
    
resp, t_prompts, t_completions = call_llm(prompt_c1)
print(f"LLM response based on prompt with {t_prompts:,} tokens and response of {t_completions:,} tokens.")
print(resp)

In [0]:
# ======= C2: Examine the least frequent process variants to identify anomalies according to your domain knowledge.
# BOTTOM 100
prompt_c2 = [
            {"role": "system", "content": system_1_role},
            {"role": "user", "content": user_1_c2},
            {"role": "user", "content": user_2_context},
            {"role": "user", "content": user_example_1},
            {"role": "assistant", "content": assistant_example_1},
            {"role": "user", "content": user_example_2},
            {"role": "assistant", "content": assistant_example_2},
            {"role": "user", "content": user_example_3},
            {"role": "assistant", "content": assistant_example_3},
            {"role": "user", "content": user_3_input},
            {"role": "user", "content": user_4_c2_corpus},
            {"role": "user", "content": user_5_c2_steps},
            {"role": "system", "content": system_2_c2_objective},
            {"role": "system", "content": system_3_narrowing},
        ]
        
resp, t_prompts, t_completions = call_llm(prompt_c2)
print(f"LLM response based on prompt with {t_prompts:,} tokens and response of {t_completions:,} tokens.")
print(resp)

In [0]:
# ======= V1: Perform a semantic clustering of the process variants and provide a summary.
prompt_v1 = [
            {"role": "system", "content": system_1_role},
            {"role": "user", "content": user_1_v1},
            {"role": "user", "content": user_2_context},
            {"role": "user", "content": user_3_input},
            {"role": "user", "content": user_4_v1_corpus},
            {"role": "user", "content": user_5_v1_steps},
            {"role": "system", "content": system_2_v1_objective},
            {"role": "system", "content": system_3_narrowing},
        ]
        
resp, t_prompts, t_completions = call_llm(prompt_v1)
print(f"LLM response based on prompt with {t_prompts:,} tokens and response of {t_completions:,} tokens.")
print(resp)